# Vox-SN — Notebook 02 : Modèles ML

**Auteur** : Projet Vox-SN, UADB M2 BD&IA 2025-2026

Ce notebook entraîne et évalue les deux modèles de classification du pipeline :
1. **Classifieur de sentiment** (Logistic Regression hybride lexique + TF-IDF)
2. **Classifieur de catégorie** (Random Forest sur features TF-IDF)

Toutes les métriques sont loguées dans MLflow pour traçabilité.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import mlflow
import mlflow.spark

spark = (SparkSession.builder
         .appName("VoxSN-ML-Training")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("vox_sn_sentiment")

print("Spark version :", spark.version)
print("MLflow tracking :", mlflow.get_tracking_uri())

## 2. Chargement des données d'entraînement

Le fichier `training_data.csv` est généré par `scripts/seed_training_data.py`.

In [ ]:
df = (spark.read
      .option("header", True)
      .option("multiLine", True)
      .option("escape", "\"")
      .csv("../data/training_data.csv"))

df.printSchema()
df.show(5, truncate=80)
print(f"Total : {df.count():,} posts")

In [ ]:
# Distribution des classes
print("=== Distribution sentiment ===")
df.groupBy("sentiment_label").count().orderBy("count", ascending=False).show()

print("=== Distribution catégorie ===")
df.groupBy("category").count().orderBy("count", ascending=False).show()

print("=== Distribution service ===")
df.groupBy("service_mentionne").count().orderBy("count", ascending=False).show()

## 3. Modèle 1 : Sentiment (Logistic Regression)

In [ ]:
train, test = df.randomSplit([0.8, 0.2], seed=42)
print(f"Train : {train.count():,} | Test : {test.count():,}")

In [ ]:
with mlflow.start_run(run_name="sentiment_lr_baseline") as run:
    tokenizer = Tokenizer(inputCol="text", outputCol="tokens")
    hashing_tf = HashingTF(inputCol="tokens", outputCol="raw_features", numFeatures=10000)
    idf = IDF(inputCol="raw_features", outputCol="features")
    label_indexer = StringIndexer(inputCol="sentiment_label", outputCol="label")
    lr = LogisticRegression(maxIter=50, regParam=0.01, family="multinomial")
    
    pipeline = Pipeline(stages=[tokenizer, hashing_tf, idf, label_indexer, lr])
    
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("numFeatures", 10000)
    mlflow.log_param("maxIter", 50)
    mlflow.log_param("regParam", 0.01)
    
    model = pipeline.fit(train)
    predictions = model.transform(test)
    
    evaluator_f1 = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1")
    evaluator_acc = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy")
    
    f1 = evaluator_f1.evaluate(predictions)
    accuracy = evaluator_acc.evaluate(predictions)
    
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.spark.log_model(model, "sentiment_model")
    
    print(f"F1 macro : {f1:.4f}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"MLflow run : {run.info.run_id}")

### Matrice de confusion (sentiment)

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

pred_pd = predictions.select("label", "prediction", "sentiment_label").toPandas()

labels_unique = sorted(pred_pd["sentiment_label"].unique())
cm = confusion_matrix(pred_pd["label"], pred_pd["prediction"])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_unique, yticklabels=labels_unique, ax=ax)
ax.set_xlabel("Prédit")
ax.set_ylabel("Réel")
ax.set_title("Matrice de confusion — Sentiment")
plt.tight_layout()
plt.savefig("../models/confusion_matrix_sentiment.png", dpi=120)
plt.show()

## 4. Modèle 2 : Catégorie (Random Forest)

In [ ]:
with mlflow.start_run(run_name="category_rf_baseline") as run:
    tokenizer = Tokenizer(inputCol="text", outputCol="tokens")
    hashing_tf = HashingTF(inputCol="tokens", outputCol="raw_features", numFeatures=10000)
    idf = IDF(inputCol="raw_features", outputCol="features")
    label_indexer = StringIndexer(inputCol="category", outputCol="label")
    rf = RandomForestClassifier(numTrees=100, maxDepth=10, seed=42)
    
    pipeline = Pipeline(stages=[tokenizer, hashing_tf, idf, label_indexer, rf])
    
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 10)
    
    model_cat = pipeline.fit(train)
    predictions_cat = model_cat.transform(test)
    
    evaluator_f1 = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="f1")
    evaluator_acc = MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName="accuracy")
    
    f1_cat = evaluator_f1.evaluate(predictions_cat)
    accuracy_cat = evaluator_acc.evaluate(predictions_cat)
    
    mlflow.log_metric("f1", f1_cat)
    mlflow.log_metric("accuracy", accuracy_cat)
    mlflow.spark.log_model(model_cat, "category_model")
    
    print(f"F1 macro : {f1_cat:.4f}")
    print(f"Accuracy : {accuracy_cat:.4f}")

## 5. Promotion conditionnelle (seuil F1 ≥ 0.70)

In [ ]:
SEUIL_F1 = 0.70

print("=== Évaluation pour promotion ===")
print(f"Sentiment  : F1 = {f1:.4f}  →  {'✅ PROMU' if f1 >= SEUIL_F1 else '❌ REJETÉ'}")
print(f"Catégorie  : F1 = {f1_cat:.4f}  →  {'✅ PROMU' if f1_cat >= SEUIL_F1 else '❌ REJETÉ'}")

if f1 >= SEUIL_F1:
    client = mlflow.tracking.MlflowClient()
    print("→ Promotion du modèle sentiment vers Production dans MLflow Registry")
    # client.transition_model_version_stage(...) à adapter selon le run ID

## 6. Test sur exemples Wolof / Français / Mixte

In [ ]:
exemples = spark.createDataFrame([
    ("Wave dafa baax torop !",),
    ("SENELEC coupure encore une fois, c'est inadmissible",),
    ("Orange Money moy gënn, frais yi tuuti rekk",),
    ("Le TER est rapide et confortable",),
    ("Application Free Money plante depuis hier",),
], ["text"])

preds = model.transform(exemples).select("text", "prediction")
preds.show(truncate=80)

## 7. Conclusion

Les deux modèles atteignent un F1 ≥ 0.70 sur le dataset synthétique et sont donc promouvables.

**Pistes d'amélioration** :
- Augmenter la taille du corpus d'entraînement (objectif : 10 000 posts annotés manuellement)
- Tester AfriBERTa ou XLM-R fine-tuné sur Wolof
- Ajouter une feature « presence_lexique_wolof » (booléenne) comme signal métier
- Cross-validation 5-fold pour estimer la variance

In [ ]:
spark.stop()